# 02 · Limpieza de datos — ENCC 2017

Este cuaderno aplica sobre el archivo crudo las decisiones del **registro de decisiones** y guarda el resultado en `datos/derivados/encc_2017_limpio.parquet`. Cada paso se verifica con `assert`, y la tabla final `pasos` documenta cuántas filas y columnas quedan después de cada uno.

El diccionario con el perfil y la decisión por variable está en `01_diccionario_encc.ipynb`.

## Registro de decisiones

| # | Qué | Qué se hizo | Evidencia | Por qué (alternativa descartada) | Pregunta |
|---|---|---|---|---|---|
| 1 | Filas 1949 y 2739 | Se descartaron | `id` = "metalicas" y nulo; `sexo` = "ALQUILADA"; región y fecha vacías; 2802 → 2800 filas | Columnas desplazadas; reubicar valores implicaría inventar datos | Todas |
| 2 | `horas_*_total` (7 columnas) | Se descartaron; se usa `minutos_*_total` | Fechas en 51–241 filas por columna y valores > 24 h; horas × 60 = minutos en el 100 % de las filas comparables | Daño por la conversión CSV→XLSX; los minutos equivalen y están limpios (alternativa descartada: reparar las horas) | P2, P4, P8 |
| 3 | `"NA"` literal (187 columnas, 18,0 % de las celdas) | Se conserva como "no aplica" y se excluye de la base de cada pregunta | `p10` = NO tiene 219 casos, igual que los `NA` de `p13_x` | Es un salto del cuestionario, no un dato perdido (alternativa descartada: tratarlo como faltante) | P1, P5, P9 |
| 4 | NS/NC en texto (todas las columnas, 7.593 celdas) | Pasan a `NaN` | Conteo en el diccionario y en el paso 3 del código | No son respuestas válidas; se excluyen de la base de cada pregunta | P5, P9, P10 |
| 5 | `p42`: 17 vacíos | Se conservan como `NaN`, no se imputan | 17 de 2800 (0,6 %) | Imputar inventaría respuestas | P3 |
| 6 | `minutos_musica_total` = 1830 | Pasa a `NaN`; la fila se conserva | 1 valor de 30,5 h/día; el máximo posible es 1440 | Valor imposible; eliminar la fila perdería el resto de sus respuestas | — |
| 7 | `pondera_dem` | Se reporta con y sin ponderar (se aplica al analizar, no acá) | Cada región tiene 400 casos por diseño; pesos de 2 a 182.969 | Sin ponderar, los porcentajes no son nacionales | P1–P10 |
| 8 | `NSEcat1` y `p106` | Se declaran categóricas ordinales | `NSEcat1`: 7 niveles; `p106`: 10 niveles, sin vacíos | El orden importa para tablas y gráficos; el orden entre "terciario" y "universitario" es a confirmar | P3, P7 |

## 1. Lectura

Todo se lee como texto, para no perder nada: el `"NA"` (no aplica) y el `NS/NC` se conservan tal cual hasta decidir qué hacer con cada uno.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos').is_dir())
RUTA = RAIZ / 'datos' / 'crudos' / 'encc_2017.xlsx'
SALIDA = RAIZ / 'datos' / 'derivados' / 'encc_2017_limpio.parquet'
SALIDA.parent.mkdir(parents=True, exist_ok=True)

enc = pd.read_excel(RUTA, dtype=str, keep_default_na=False, na_values=[''])
pasos = [('lectura del crudo', len(enc), enc.shape[1])]

## 2. Filas corruptas (registro #1)

In [3]:
enc = enc[enc['sexo'] != 'ALQUILADA'].reset_index(drop=True)
assert len(enc) == 2800
assert enc['id'].notna().all() and enc['id'].is_unique
pasos.append(('se descartan filas ALQUILADA', len(enc), enc.shape[1]))

## 3. Columnas dañadas (registro #2)

In [4]:
horas = [c for c in enc.columns if c.startswith('horas_') and c.endswith('_total')]
assert len(horas) == 7
enc = enc.drop(columns=horas)
assert not any(c in enc.columns for c in horas)
pasos.append(('se descartan horas_*_total', len(enc), enc.shape[1]))

## 4. NS/NC (registro #4)

El `NS/NC` pasa a `NaN`. El `"NA"` **no se toca**: es "no aplica" por salto del cuestionario (registro #3).

In [5]:
antes_nsnc = int((enc == 'NS/NC').sum().sum())
enc = enc.mask(enc == 'NS/NC')
assert int((enc == 'NS/NC').sum().sum()) == 0
pasos.append((f'NS/NC a NaN ({antes_nsnc} celdas)', len(enc), enc.shape[1]))

## 5. Tipos numéricos y valores imposibles (registros #5 y #6)

`errors='raise'` hace que falle si alguna celda no es numérica. Un día tiene 1440 minutos: nada puede pasar de ahí.

In [6]:
a_numero = ['edad', 'pondera_dem', 'minutos_internet_total', 'minutos_redes_total',
            'minutos_videojuegos_total', 'minutos_musica_total']
for c in a_numero:
    enc[c] = pd.to_numeric(enc[c], errors='raise')
enc['edad'] = enc['edad'].astype('Int64')
assert enc['edad'].between(13, 92).all()

imposibles = enc['minutos_musica_total'] > 1440
assert imposibles.sum() == 1
enc.loc[imposibles, 'minutos_musica_total'] = np.nan
assert enc['minutos_musica_total'].max() <= 1440
for c in ['minutos_internet_total', 'minutos_redes_total', 'minutos_videojuegos_total']:
    assert enc[c].dropna().between(0, 1440).all()

assert enc['p42'].isna().sum() == 17   # registro #5: se conservan, no se imputan

## 6. Variables ordinales (registro #8)

In [7]:
orden_nse = ['AB', 'C1', 'C2', 'C3', 'D1', 'D2', 'E']
orden_estudios = ['SIN ESTUDIOS', 'PRIMARIA INCOMPLETA', 'PRIMARIA COMPLETA',
                  'SECUNDARIA INCOMPLETA', 'SECUNDARIA COMPLETA',
                  'TERCIARIO INCOMPLETO', 'TERCIARIO COMPLETO',
                  'UNIVERSITARIO INCOMPLETO', 'UNIVERSITARIO COMPLETO',
                  'POSGRADO INCOMPLETO O COMPLETO']

assert enc['NSEcat1'].dropna().isin(orden_nse).all() and enc['NSEcat1'].isna().sum() == 2
assert enc['p106'].isin(orden_estudios).all()      # sin vacíos y sin categorías fuera de la lista

enc['NSEcat1'] = pd.Categorical(enc['NSEcat1'], categories=orden_nse, ordered=True)
enc['p106']    = pd.Categorical(enc['p106'], categories=orden_estudios, ordered=True)

## 7. Verificación final y guardado

In [8]:
assert len(enc) == 2800 and enc['region'].value_counts().eq(400).all()
enc.to_parquet(SALIDA, index=False)

leido = pd.read_parquet(SALIDA)
assert leido.shape == enc.shape
assert leido['NSEcat1'].cat.ordered and leido['p106'].cat.ordered
pasos.append(('guardado en derivados', len(leido), leido.shape[1]))

pd.DataFrame(pasos, columns=['paso', 'filas', 'columnas'])

,paso,filas,columnas
0,lectura del crudo,2802,450
1,se descartan filas ALQUILADA,2800,450
2,se descartan horas_*_total,2800,443
3,NS/NC a NaN (7593 celdas),2800,443
4,guardado en derivados,2800,443
